## 1. Imports

In [1]:
import os
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

## 2. Paths

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/FinSight_AI"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

PROCESSED_PATH = os.path.join(
    DATA_PATH,
    "processed"
)

CLASSIFICATION_PATH = os.path.join(
    DATA_PATH,
    "classification"
)

RULE_PATH = os.path.join(
    CLASSIFICATION_PATH,
    "rule_classification"
)

SEMANTIC_PATH = os.path.join(
    CLASSIFICATION_PATH,
    "semantic_classification"
)

HYBRID_PATH = os.path.join(
    CLASSIFICATION_PATH,
    "hybrid_classification"
)

os.makedirs(
    HYBRID_PATH,
    exist_ok=True
)

## 3 . Configuration

In [3]:
RULE_FILE = os.path.join(

    RULE_PATH,

    "news_classification_rule.parquet"

)

SEMANTIC_FILE = os.path.join(

    SEMANTIC_PATH,

    "semantic_classification.parquet"

)

BATCH_SIZE = 5000

SEMANTIC_CONFIDENCE_THRESHOLD = 0.70

## 4. Load Semantic Dataset

In [5]:
semantic_df = pd.read_parquet(

    SEMANTIC_FILE

)

print("Semantic Dataset")
print(semantic_df.shape)

semantic_df.head()

Semantic Dataset
(1584926, 5)


,news_id,headline,semantic_event,semantic_confidence,top_k_predictions
0,18,"Q1 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7577,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."
1,19,Pershing Square 13F Shows Fund Raises Stake In...,Funding,0.7031,"[{""event"": ""Funding"", ""confidence"": 0.7031}, {..."
2,20,How Bill Ackman Successfully Navigated Coronav...,Technical Analysis,0.7333,"[{""event"": ""Technical Analysis"", ""confidence"":..."
3,25,Agilent Reports Has Become Top-Level Sponsor O...,ESG,0.7724,"[{""event"": ""ESG"", ""confidence"": 0.7724}, {""eve..."
4,54,"Q4 13F Roundup: How Buffett, Einhorn, Ackman A...",Analyst Rating,0.7478,"[{""event"": ""Analyst Rating"", ""confidence"": 0.7..."


## 5. Build Fast Semantic Lookup

In [6]:
semantic_lookup = semantic_df.set_index(

    "news_id"

)[

    [

        "semantic_event",

        "semantic_confidence",

        "top_k_predictions"

    ]

].to_dict(

    orient="index"

)

del semantic_df

gc.collect()
print("Semantic Lookup Created")
print(f"Entries : {len(semantic_lookup):,}")

Semantic Lookup Created
Entries : 1,584,926


## 6. Output Folder

In [7]:
HYBRID_BATCH_PATH = os.path.join(

    HYBRID_PATH,

    "hybrid_batches"

)

os.makedirs(

    HYBRID_BATCH_PATH,

    exist_ok=True

)

print(HYBRID_BATCH_PATH)

/content/drive/MyDrive/FinSight_AI/data/classification/hybrid_classification/hybrid_batches


## 7. Checkpoint Functions

In [8]:
CHECKPOINT_FILE = os.path.join(

    HYBRID_PATH,

    "hybrid_checkpoint.json"

)

def save_checkpoint(

    batch_number,

    processed_rows

):

    checkpoint = {

        "last_batch": batch_number,

        "processed_rows": processed_rows

    }

    with open(

        CHECKPOINT_FILE,

        "w"

    ) as f:

        json.dump(

            checkpoint,

            f,

            indent=4

        )


def load_checkpoint():

    if os.path.exists(

        CHECKPOINT_FILE

    ):

        with open(

            CHECKPOINT_FILE,

            "r"

        ) as f:

            return json.load(f)

    return {

        "last_batch": 0,

        "processed_rows": 0

    }

## 8. Hybrid Decision Function

In [9]:
def hybrid_decision(row):

    semantic = semantic_lookup.get(

        row["news_id"],

        None

    )

    # Rule Engine succeeded
    if row["event"] != "Other":

        return {

            "final_event": row["event"],

            "final_confidence": row["confidence"],

            "classification_source": "Rule"

        }

    # No semantic prediction available
    if semantic is None:

        return {

            "final_event": "Other",

            "final_confidence": row["confidence"],

            "classification_source": "Unknown"

        }

    # Semantic prediction available
    if semantic["semantic_confidence"] >= SEMANTIC_CONFIDENCE_THRESHOLD:

        return {

            "final_event": semantic["semantic_event"],

            "final_confidence": semantic["semantic_confidence"],

            "classification_source": "Semantic"

        }

    return {

        "final_event": "Other",

        "final_confidence": semantic["semantic_confidence"],

        "classification_source": "Unknown"

    }

## 9. Process One Batch

In [10]:
def process_batch(batch_df):

    final_events = []

    final_confidences = []

    sources = []

    for _, row in batch_df.iterrows():

        result = hybrid_decision(row)

        final_events.append(

            result["final_event"]

        )

        final_confidences.append(

            result["final_confidence"]

        )

        sources.append(

            result["classification_source"]

        )

    output = batch_df.copy()

    output["final_event"] = final_events

    output["final_confidence"] = final_confidences

    output["classification_source"] = sources

    return output

## 10. Test Batch Reader

In [13]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(RULE_FILE)

print("Rule Dataset")

print(f"Rows : {parquet_file.metadata.num_rows:,}")
print(f"Row Groups : {parquet_file.num_row_groups}")

Rule Dataset
Rows : 3,215,296
Row Groups : 4


## 11. Read One Batch

In [14]:
batch_iterator = parquet_file.iter_batches(

    batch_size=BATCH_SIZE

)

first_batch = next(batch_iterator)

test_batch = first_batch.to_pandas()

print(test_batch.shape)

display(test_batch.head())

(5000, 8)


,news_id,headline,event,market_signal,confidence,matched_keywords,rule_score,method
0,1,Stocks That Hit 52-Week Highs On Friday,Market Movement,Neutral,1.0,52[- ]week\s+highs?,5.0,Rule
1,2,Stocks That Hit 52-Week Highs On Wednesday,Market Movement,Neutral,1.0,52[- ]week\s+highs?,5.0,Rule
2,3,71 Biggest Movers From Friday,Market Movement,Neutral,1.0,"biggest movers, biggest\s+movers",10.0,Rule
3,4,46 Stocks Moving In Friday's Mid-Day Session,Market Movement,Neutral,1.0,"mid-day session, mid[- ]day\s+session, stocks ...",14.0,Rule
4,5,B of A Securities Maintains Neutral on Agilent...,Analyst Rating,Bullish,1.0,"maintains, maintains?\s+\w+\s+on, neutral, pri...",13.0,Rule


## 12. Process One Batch

In [15]:
test_output = process_batch(test_batch)

display(test_output.head())

,news_id,headline,event,market_signal,confidence,matched_keywords,rule_score,method,final_event,final_confidence,classification_source
0,1,Stocks That Hit 52-Week Highs On Friday,Market Movement,Neutral,1.0,52[- ]week\s+highs?,5.0,Rule,Market Movement,1.0,Rule
1,2,Stocks That Hit 52-Week Highs On Wednesday,Market Movement,Neutral,1.0,52[- ]week\s+highs?,5.0,Rule,Market Movement,1.0,Rule
2,3,71 Biggest Movers From Friday,Market Movement,Neutral,1.0,"biggest movers, biggest\s+movers",10.0,Rule,Market Movement,1.0,Rule
3,4,46 Stocks Moving In Friday's Mid-Day Session,Market Movement,Neutral,1.0,"mid-day session, mid[- ]day\s+session, stocks ...",14.0,Rule,Market Movement,1.0,Rule
4,5,B of A Securities Maintains Neutral on Agilent...,Analyst Rating,Bullish,1.0,"maintains, maintains?\s+\w+\s+on, neutral, pri...",13.0,Rule,Analyst Rating,1.0,Rule


## 13. Process Entire Dataset

In [16]:
parquet_file = pq.ParquetFile(RULE_FILE)

checkpoint = load_checkpoint()

START_BATCH = checkpoint["last_batch"]

processed_rows = checkpoint["processed_rows"]

overall_start = time.time()

batch_iterator = parquet_file.iter_batches(

    batch_size=BATCH_SIZE

)

print("Hybrid Classification")

for batch_number, batch in enumerate(

    tqdm(

        batch_iterator,

        total=(parquet_file.metadata.num_rows + BATCH_SIZE - 1) // BATCH_SIZE,

        desc="Hybrid"

    )

):

    if batch_number < START_BATCH:

        continue

    output_file = os.path.join(

        HYBRID_BATCH_PATH,

        f"hybrid_batch_{batch_number+1:04d}.parquet"

    )

    if os.path.exists(output_file):

        continue

    batch_df = batch.to_pandas()

    batch_result = process_batch(batch_df)

    batch_result.to_parquet(

        output_file,

        index=False

    )

    processed_rows += len(batch_result)

    save_checkpoint(

        batch_number + 1,

        processed_rows

    )

    if (

        (batch_number + 1) % 25 == 0

        or

        batch_number == ((parquet_file.metadata.num_rows + BATCH_SIZE - 1) // BATCH_SIZE) - 1

    ):

        elapsed = time.time() - overall_start

        avg_batch = elapsed / (batch_number + 1)

        eta = (

            ((parquet_file.metadata.num_rows + BATCH_SIZE - 1) // BATCH_SIZE)

            - batch_number

            - 1

        ) * avg_batch / 60

        print()

        print("=" * 70)

        print(f"Batch : {batch_number+1}")

        print(f"Rows Processed : {processed_rows:,}")

        print(f"ETA : {eta:.2f} minutes")

    del batch_df
    del batch_result

    gc.collect()

print()

print("HYBRID CLASSIFICATION COMPLETED")

print(f"Processed Rows : {processed_rows:,}")

Hybrid Classification


Hybrid:   0%|          | 0/644 [00:00<?, ?it/s]


Batch : 25
Rows Processed : 125,000
ETA : 5.50 minutes

Batch : 50
Rows Processed : 250,000
ETA : 5.08 minutes

Batch : 75
Rows Processed : 375,000
ETA : 4.97 minutes

Batch : 100
Rows Processed : 500,000
ETA : 4.65 minutes

Batch : 125
Rows Processed : 625,000
ETA : 4.39 minutes

Batch : 150
Rows Processed : 750,000
ETA : 4.12 minutes

Batch : 175
Rows Processed : 875,000
ETA : 3.84 minutes

Batch : 200
Rows Processed : 1,000,000
ETA : 3.62 minutes

Batch : 225
Rows Processed : 1,125,000
ETA : 3.42 minutes

Batch : 250
Rows Processed : 1,250,000
ETA : 3.21 minutes

Batch : 275
Rows Processed : 1,375,000
ETA : 3.00 minutes

Batch : 300
Rows Processed : 1,500,000
ETA : 2.77 minutes

Batch : 325
Rows Processed : 1,625,000
ETA : 2.57 minutes

Batch : 350
Rows Processed : 1,750,000
ETA : 2.37 minutes

Batch : 375
Rows Processed : 1,875,000
ETA : 2.20 minutes

Batch : 400
Rows Processed : 2,000,000
ETA : 2.01 minutes

Batch : 425
Rows Processed : 2,125,000
ETA : 1.81 minutes

Batch : 450
R

## 14. Merge All Hybrid Batches

In [17]:
hybrid_files = sorted(

    os.listdir(HYBRID_BATCH_PATH)

)

hybrid_df = pd.concat(

    (

        pd.read_parquet(

            os.path.join(

                HYBRID_BATCH_PATH,

                file

            )

        )

        for file in tqdm(

            hybrid_files,

            desc="Merging Hybrid Batches"

        )

    ),

    ignore_index=True

)

OUTPUT_FILE = os.path.join(

    HYBRID_PATH,

    "news_classification_final.parquet"

)

hybrid_df.to_parquet(

    OUTPUT_FILE,

    index=False

)

print("MASTER HYBRID DATASET CREATED")

print(hybrid_df.shape)

Merging Hybrid Batches:   0%|          | 0/644 [00:00<?, ?it/s]

MASTER HYBRID DATASET CREATED
(3215296, 11)


## 15. Evaluation

In [18]:
event_summary = (

    hybrid_df

    .groupby("final_event")

    .agg(

        headlines=("news_id","count"),

        avg_confidence=("final_confidence","mean")

    )

    .sort_values(

        "headlines",

        ascending=False

    )

    .reset_index()

)

display(event_summary)

,final_event,headlines,avg_confidence
0,Product Launch,654857,0.758737
1,Earnings,567435,0.944125
2,Market Movement,469086,0.833923
3,Analyst Rating,277248,0.899277
4,Merger & Acquisition,231793,0.790391
5,Commodity,176547,0.840405
6,Other,137156,0.678070
7,Technical Analysis,113179,0.786142
8,Dividend,110494,0.956925
9,Executive Change,45456,0.917445


## 16. Summary

In [19]:
summary = pd.DataFrame({

    "Metric":[

        "Total Headlines",

        "Unique Events",

        "Rule Classified",

        "Semantic Classified",

        "Unknown",

        "Output File"

    ],

    "Value":[

        len(hybrid_df),

        hybrid_df["final_event"].nunique(),

        (hybrid_df["classification_source"]=="Rule").sum(),

        (hybrid_df["classification_source"]=="Semantic").sum(),

        (hybrid_df["classification_source"]=="Unknown").sum(),

        os.path.basename(OUTPUT_FILE)

    ]

})

display(summary)

,Metric,Value
0,Total Headlines,3215296
1,Unique Events,30
2,Rule Classified,1630370
3,Semantic Classified,1447770
4,Unknown,137156
5,Output File,news_classification_final.parquet
